# Лабораторная работа №5. Градиентный бустинг

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются те же Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE), что в [лабораторной №1](lab1.ipynb). Тест и фолды не меняются. Основные метрики подбора: macro-F1 вверх, MAE вниз.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует исходные числовые признаки.

In [1]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'raw'

def sklearn_model(task, params):
    if task == 'classification':
        return GradientBoostingClassifier(random_state=SEED, **params)
    return GradientBoostingRegressor(random_state=SEED, **params)

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [2]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.9108,0.9205,0.0903,NaN,NaN
1,regression,NaN,NaN,0.1363,86.3128,160.6849


## Гипотезы и кросс-валидация

Проверяю глубину слабого дерева, шаг и число итераций: небольшие шаги требуют больше деревьев, а слишком глубокие деревья повышают риск переобучения. Как и в лесу, новые признаки проверяются отдельно и вместе с настройкой параметров.

В обоих случаях 5 фолдов. Сравниваю препроцессинг без подбора, подбор без изменения признаков и их комбинацию. Сначала выбираю лучшую альтернативу бейзлайну на CV, затем оцениваю её на отложенной части. `cv_std` показывает разброс оценок по фолдам, это не доверительный интервал.

In [3]:
feature_options = ['features']
class_grid = [{'n_estimators': 80, 'learning_rate': 0.05, 'max_depth': 2}, {'n_estimators': 140, 'learning_rate': 0.05, 'max_depth': 3}, {'n_estimators': 150, 'learning_rate': 0.08, 'max_depth': 4}, {'n_estimators': 130, 'learning_rate': 0.1, 'max_depth': 2}, {'n_estimators': 180, 'learning_rate': 0.1, 'max_depth': 3}, {'n_estimators': 220, 'learning_rate': 0.12, 'max_depth': 4}]
reg_grid = [{'n_estimators': 80, 'learning_rate': 0.05, 'max_depth': 2}, {'n_estimators': 140, 'learning_rate': 0.05, 'max_depth': 3}, {'n_estimators': 150, 'learning_rate': 0.08, 'max_depth': 4}, {'n_estimators': 130, 'learning_rate': 0.1, 'max_depth': 2}, {'n_estimators': 180, 'learning_rate': 0.1, 'max_depth': 3}, {'n_estimators': 220, 'learning_rate': 0.12, 'max_depth': 4}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

classification


,experiment,prep,params,cv_mean,cv_std
0,baseline,raw,{},0.9103,0.0198
1,parameters,raw,"{'n_estimators': 140, 'learning_rate': 0.05, '...",0.9069,0.0242
2,combined,features,"{'n_estimators': 130, 'learning_rate': 0.1, 'm...",0.9014,0.0231
3,preprocessing,features,{},0.8973,0.0141


regression


,experiment,prep,params,cv_mean,cv_std
0,combined,features,"{'n_estimators': 220, 'learning_rate': 0.12, '...",35.4640,2.2592
1,preprocessing,features,{},54.9888,2.9942
2,parameters,raw,"{'n_estimators': 220, 'learning_rate': 0.12, '...",68.1113,3.1993
3,baseline,raw,{},92.4641,8.1676


Лучший вариант из альтернатив заново обучаю на всём train. Прирост на CV не гарантирует прироста на тесте.

In [4]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.9108,0.9205,0.1316,NaN,NaN
1,regression,NaN,NaN,0.5470,33.3555,66.4437


## Самописная реализация

Регрессор стартует со среднего и учит деревья на остатках y − F. Классификатор стартует с логита доли положительного класса, строит дерево на y − p и делает ньютоновскую поправку в каждом листе: сумма остатков делится на сумму p(1 − p). Прогноз этапа прибавляется к ансамблю с множителем `learning_rate`.

Самописный алгоритм проверяется с обоими рецептами sklearn: исходным и выбранным по CV. Приёмы обработки признаков общие, чтобы сравнить реализацию модели, а не разные входные данные.

In [5]:
from models.boosting import Boosting

def own_model(task, params):
    return Boosting(task, random_state=SEED, **params)

own_base = {task: evaluate(task, own_model(task, {}), BASE) for task in TASKS}
own_improved = {task: evaluate(task, own_model(task, row.params), row.prep)
                for task, row in chosen.items()}

Сохраняю четыре сравнимые строки на задачу. `Δ` ниже означает разницу с соответствующим библиотечным или исходным вариантом, а не процентное изменение. Для MAE отрицательная разница означает улучшение.

In [6]:
versions = {'sklearn baseline': baseline, 'sklearn improved': improved,
            'own baseline': own_base, 'own improved': own_improved}
scores = pd.DataFrame([dict(version=version, **metrics)
                       for version, outcomes in versions.items()
                       for metrics, _ in outcomes.values()])
from pathlib import Path

Path('results').mkdir(exist_ok=True)
scores.drop(columns='fit_seconds').to_csv('results/lab5.csv', index=False)
for task, metric in zip(TASKS, ('f1_macro', 'mae_ms')):
    subset = scores.query('task == @task').dropna(axis=1).set_index('version')
    display(subset.drop(columns='task').round(4))
    s = subset[metric]
    display(pd.Series({'sklearn improved − sklearn baseline': s.iloc[1] - s.iloc[0],
                       'own baseline − sklearn baseline': s.iloc[2] - s.iloc[0],
                       'own improved − sklearn improved': s.iloc[3] - s.iloc[1],
                       'own improved − own baseline': s.iloc[3] - s.iloc[2]},
                      name=f'Δ {metric}').round(4))

,f1_macro,accuracy,fit_seconds
version,,,
sklearn baseline,0.9108,0.9205,0.0903
sklearn improved,0.9108,0.9205,0.1316
own baseline,0.9108,0.9205,0.1739
own improved,0.9108,0.9205,0.2752


sklearn improved − sklearn baseline    0.0
own baseline − sklearn baseline        0.0
own improved − sklearn improved        0.0
own improved − own baseline            0.0
Name: Δ f1_macro, dtype: float64

,fit_seconds,mae_ms,rmse_ms
version,,,
sklearn baseline,0.1363,86.3128,160.6849
sklearn improved,0.5470,33.3555,66.4437
own baseline,0.6382,86.3128,160.6849
own improved,3.1026,33.2128,66.1954


sklearn improved − sklearn baseline   -52.9573
own baseline − sklearn baseline         0.0000
own improved − sklearn improved        -0.1427
own improved − own baseline           -53.1000
Name: Δ mae_ms, dtype: float64

Сравниваю ошибки обеих улучшенных моделей на тех же тестовых объектах. Матрица показывает, какие классы путаются. В регрессии сравниваю величину ошибки на быстрых и медленных конфигурациях; медианный порог беру из train.

In [7]:
from sklearn.metrics import confusion_matrix

_, _, _, test_channel = split('classification')
_, _, train_time, test_time = split('regression')
bands = np.where(test_time <= train_time.median(), 'быстрые', 'медленные')
for label, outcomes in [('sklearn', improved), ('самописная', own_improved)]:
    print(label)
    display(pd.DataFrame(confusion_matrix(test_channel, outcomes['classification'][1], labels=[1, 2]),
                         index=['факт Horeca', 'факт Retail'],
                         columns=['прогноз Horeca', 'прогноз Retail']))
    error = np.abs(np.asarray(test_time) - outcomes['regression'][1])
    display(pd.Series(error).groupby(bands).agg(['size', 'mean']).round(2))

sklearn


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,2,26


,size,mean
быстрые,308,12.83
медленные,292,55.01


самописная


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,2,26


,size,mean
быстрые,308,12.86
медленные,292,54.68


## Выводы

- **Классификация.** Исходный бустинг остался первым на CV (macro-F1 0,9103 против 0,9069 у лучшей альтернативы). На тесте все четыре версии дали 0,9108. Название `improved` здесь обозначает проверенный настроенный рецепт, а не реальный прирост; оставил бы бейзлайн.
- **Регрессия.** Все три альтернативы снизили CV MAE. Лучшей оказалась комбинация новых признаков с 220 деревьями глубины 4 и шагом 0,12: 35,46 мс против 92,46 мс. На тесте MAE sklearn упала с 86,31 до 33,36 мс. Бейзлайны совпали с точностью до округления, после настройки самописный бустинг получил 33,21 мс. Разница 0,14 мс мала относительно разброса по фолдам.